In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from xarray.backends.file_manager import CachingFileManager
import os
from netCDF4 import Dataset
import time

final_path = "C:\\Users\\NEA-ARN\\Documents\\quant\\quant-regime-research\\data\\processed\\SPY_cond_ex_tab.nc"

def build_probability_tensor(P, taus, q_vals, r_vals):
    directions = ["+", "-"]

    prob = np.zeros((2, len(taus), len(q_vals), len(r_vals)))
    prob_se = np.zeros_like(prob)  # <-- NEW
    joint_ct = np.zeros_like(prob, dtype=int)
    cond_ct = np.zeros((2, len(taus), len(q_vals)), dtype=int)

    for d_i, direction in enumerate(directions):
        for t_i, tau in enumerate(taus):

            inc = P[tau:] - P[:-tau]
            inc_now = inc[:-tau]
            inc_future = inc[tau:]

            for q_i, q in enumerate(q_vals):

                if direction == "+":
                    q_thresh = np.quantile(inc, q)
                    cond_event = inc_now >= q_thresh
                else:
                    q_thresh = np.quantile(inc, 1 - q)
                    cond_event = inc_now <= q_thresh

                cond_count = cond_event.sum()
                cond_ct[d_i, t_i, q_i] = cond_count

                if cond_count == 0:
                    prob[d_i, t_i, q_i, :] = np.nan
                    prob_se[d_i, t_i, q_i, :] = np.nan
                    continue

                for r_i, r in enumerate(r_vals):

                    if direction == "+":
                        r_thresh = np.quantile(inc, r)
                        future_event = inc_future >= r_thresh
                    else:
                        r_thresh = np.quantile(inc, 1 - r)
                        future_event = inc_future <= r_thresh

                    joint = cond_event & future_event
                    joint_count = joint.sum()

                    joint_ct[d_i, t_i, q_i, r_i] = joint_count

                    p_hat = joint_count / cond_count
                    prob[d_i, t_i, q_i, r_i] = p_hat

                    # Bernoulli standard error
                    prob_se[d_i, t_i, q_i, r_i] = 1/cond_count

    return xr.Dataset(
        data_vars=dict(
            prob=(["direction", "tau", "q", "r"], prob),
            prob_se=(["direction", "tau", "q", "r"], prob_se),  # <-- NEW
            joint_ct=(["direction", "tau", "q", "r"], joint_ct),
            cond_ct=(["direction", "tau", "q"], cond_ct),
        ),
        coords=dict(
            direction=directions,
            tau=taus,
            q=q_vals,
            r=r_vals,
        )
    )


In [35]:
# Load the data
df = pd.read_csv('../data/raw/SPY_daily_yahoo_raw.csv', skiprows=[0,1,2], header=None, names=['Date','Adj Close','Close','High','Low','Open','Volume'], index_col=0, parse_dates=True, dtype={'Adj Close': float, 'Close': float, 'High': float, 'Low': float, 'Open': float, 'Volume': int})

# Keep only data up to end of 2024
df = df[df.index <= '2024-12-31']

P = df['Adj Close'].values
taus = np.arange(1, 4000, 1)
q_vals = np.arange(0.01, 0.99, 0.01)
r_vals = np.arange(0.8, 0.99, 0.05)
ds = build_probability_tensor(P, taus, q_vals, r_vals)

In [36]:
q_vals

array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11,
       0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22,
       0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33,
       0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44,
       0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55,
       0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66,
       0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76, 0.77,
       0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88,
       0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98])

In [37]:
r_vals

array([0.8 , 0.85, 0.9 , 0.95])

In [38]:
prob_up = ds.prob.sel(
    direction="+",
    tau=200,
    q=0.3,
    r=0.8
).item()

prob_up

0.22417707150964813

In [39]:
ds.to_netcdf(final_path)

In [40]:
ds = xr.open_dataset(final_path)

In [41]:
prob_up = ds.prob.sel(
    direction="+",
    tau=200,
    q=0.3,
    r=0.8
).item()

prob_up_se = ds.prob_se.sel(
    direction="+",
    tau=200,
    q=0.3,
    r=0.8
).item()

print('probability estimate = '+str(round(prob_up,4))) 
print('uncertainty = '+str(round(prob_up_se,4)))

probability estimate = 0.2242
uncertainty = 0.0002


In [43]:
ds.close()  # close file after reading